In [ ]:
from src.visualiser import set_up_visualisation, visualise_fancy
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

In [ ]:
out_path = "/path/to/out_folder"
dataset_type = "mds"
input_dim = 128
confidence = 0.7

label = 0
i = "0000"
split = "test"
path = f"/path/to/dataset/Multi-DSprites/{split}/class_{label}_{i}.png"
# path = f"/path/to/dataset/CLEVR-Hans7/{split}/images/CLEVR_Hans_classid_{label}_{i}.png"
# path = f"/path/to/dataset/CLEVR-Hans3/{split}/images/CLEVR_Hans_classid_{label}_{i}.png"

In [ ]:
env_path = out_path + "/.env"
checkpoint_dir = out_path + "/checkpoints/cg/"
sa_checkpoint_dir = out_path + "/checkpoints/sa/"

In [ ]:
device = 'cuda'
env, cg, sa = set_up_visualisation(env_path, checkpoint_dir, sa_checkpoint_dir)
cg.eval()
sa.eval()

env.confidence_threshold = confidence
''

In [ ]:
image = Image.open(path).convert("RGB")
display_image = image
if dataset_type in ['ch3', 'ch7']:
    n = 256 * 0.004296875
    h, w = int(320 * n), int(480 * n)
    transform = transforms.Compose(
        [
            transforms.Resize((h, w), antialias=None),
            transforms.CenterCrop(256),
        ]
    )

    display_image = transform(image)

plt.axis("off")
plt.imshow(display_image)

In [ ]:
n = input_dim * 0.004296875
h, w = int(320 * n), int(480 * n)
if dataset_type in ['ch3', 'ch7']:
    transform = transforms.Compose(
        [
            transforms.Resize((h, w), antialias=None),
            transforms.CenterCrop(input_dim),
            transforms.PILToTensor(),
            transforms.Lambda(lambda image: (image - 127.5) / 127.5),
        ]
    )
elif dataset_type == 'mds':
    transform = transforms.Compose(
        [
            transforms.PILToTensor(),
            transforms.Lambda(lambda image: (image - 127.5) / 127.5),
        ]
    )
transformed_image = transform(image).cuda()

In [ ]:
with torch.no_grad():
    _, slot_imgs, masks, slots, attn = sa(transformed_image.unsqueeze(0))

In [ ]:
ch7_classes = {
    0: 'Large (gray) cube and large \ncylinder',
    1: 'Small metal cube and Small \n(metal) sphere',
    2: '(Small) cyan (cube) in \nfront of two red objects',
    3: 'Small green obj. and small \nbrown obj. and small purple obj. \nand two other small obj.s',
    4: '3 spheres on left side or \n3 spheres on left side and 3 metal cyl. on the right side',
    5: 'Three metal cylinders on \nright side',
    6: 'Large blue sphere and small \nyellow sphere',
}

ch3_classes = {
    0: 'Large (gray) cube and \nlarge cylinder',
    1: 'Small metal cube and \nsmall (metal) sphere',
    2: 'Large blue sphere and \nsmall yellow sphere',
}

mds_classes = {
    0: "A red square and a \nheart.",
    1: "Two hearts on the \nleft side.",
    2: "An ellipse and two \nsquares.",
    3: "A bright object infront \nof a dark background",
    4: "Three different shapes \non the right side."
}

if dataset_type == 'mds':
    class_names = mds_classes
elif dataset_type == 'ch3':
    class_names = ch3_classes
elif dataset_type == 'ch7':
    class_names = ch7_classes

visualise_fancy(env, cg, (transformed_image.cpu().permute(1,2,0) + 1) / 2, label, slots.to(device), attn, slot_imgs, masks, class_names)